# The Outlier: smolagents, Where the Action Is CodeEvery notebook so far has shared one assumption so completely that you may not havenoticed it was an assumption: **the model acts by emitting JSON**. A tool call is a nameand a dictionary of arguments. Your loop, Agent Framework, CrewAI and LangGraph disagreeabout a great deal, but all four agree about that.smolagents does not. Its `CodeAgent` writes **Python**, which is then executed, with yourtools available as ordinary functions in scope. The action space stops being a list ofschemas and becomes a programming language.That means the agent can do in one step what the others need four turns for — loop overfiles, branch on a result, compose two tools, store an intermediate value in a variable.It also means something is executing model-written code on your machine, which is therest of this notebook's subject.**What you need:** the same `.env` with your `OPENROUTER_API_KEY`.

In [ ]:
import osimport subprocessimport sysfrom pathlib import Pathfrom dotenv import find_dotenv, load_dotenvfrom smolagents import CodeAgent, OpenAIServerModel, toolfrom smolagents.models import ChatMessage# ":free" is OpenRouter's no-cost, rate-limited endpoint for this model.# Drop the suffix for the paid endpoint if the rate limit gets in your way.MODEL = "nvidia/nemotron-3.5-lightning:free"WORKDIR = Path.cwd()load_dotenv(find_dotenv())model = OpenAIServerModel(    model_id=MODEL,    api_base="https://openrouter.ai/api/v1",    api_key=os.environ["OPENROUTER_API_KEY"],)print("model   :", MODEL)print("workdir :", WORKDIR)

## 1. Does the API work?`model.generate()` is the plain request underneath everything else — no agent, no codeexecution.

In [ ]:
reply = model.generate([ChatMessage(role="user", content="Write a haiku about debugging code.")])print(reply.content)print("\nusage:", reply.token_usage)

## 2. The taskSame bug, same tests, fifth time.> **Re-run the `%%writefile buggy.py` cell to put the bug back** and start clean.

In [ ]:
%%writefile buggy.pydef add_reading(reading, log=[]):    """Append a sensor reading to a log and return the log."""    log.append(reading)    return logdef average(readings):    """Return the mean of a list of readings."""    return sum(readings) / len(readings)

In [ ]:
%%writefile test_buggy.pyfrom buggy import add_reading, averagedef test_average():    assert average([2, 4, 6]) == 4def test_logs_are_independent():    first = add_reading(1)    second = add_reading(2)    assert first == [1]    assert second == [2]

In [ ]:
print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 3. The toolsSame three functions. **Fourth framework, fourth spelling** of parameter descriptions —smolagents reads them from an `Args:` section of the docstring, with no flag required:| | how you describe a parameter ||---|---|| by hand | you wrote the JSON || Agent Framework | `Annotated[str, "..."]` || CrewAI | `Field(..., description="...")` || LangGraph | `Args:` docstring **plus** `parse_docstring=True` || smolagents | `Args:` docstring, and it will refuse to build the tool without one |smolagents is the only one of the five that treats a missing description as an errorrather than as your problem later. Given how many silent failures the other spellingsallow, that is a defensible design choice.One more difference worth noticing: because the agent writes code, these functions arenot "tools the model requests" so much as **an API the model programs against**.

In [ ]:
@tooldef read_file(path: str) -> str:    """Read the full contents of a file in the working directory.    Args:        path: File to read, e.g. buggy.py    """    return (WORKDIR / path).read_text()@tooldef edit_file(path: str, old: str, new: str) -> str:    """Replace an exact snippet of text in a file.    Args:        path: File to edit, e.g. buggy.py        old: Exact text to replace. Must appear exactly once, whitespace included.        new: Replacement text.    """    p = WORKDIR / path    text = p.read_text()    if text.count(old) == 0:        return "ERROR: 'old' not found in the file. Read it again and match it exactly."    if text.count(old) > 1:        return "ERROR: 'old' appears more than once. Include more surrounding context."    p.write_text(text.replace(old, new))    return f"ok, edited {path}"@tooldef run_tests() -> str:    """Run the pytest suite and return its output."""    result = subprocess.run(        [sys.executable, "-m", "pytest", "-q"],        capture_output=True, text=True, timeout=60, cwd=WORKDIR,    )    return (result.stdout + result.stderr)[-2000:] or "(no output)"print(edit_file.description)print(edit_file.inputs)

## 4. The agent — this is the part you writeAlmost nothing to it. `CodeAgent` takes the tools and the model, and `max_steps` is theturn cap you have set explicitly in every notebook so far.`verbosity_level=2` prints the code the model writes at each step, which is the wholeshow — watch for it.Two things `CodeAgent` does that are worth knowing before you run it:- **It adds a tool you did not write.** `final_answer` is injected automatically; calling  it is how the agent signals it is done. Print `agent.tools` after building and you will  see four, not three.- **It does not use bare `exec`.** Model-written code runs in a restricted interpreter  that blocks imports unless you allow them explicitly. That interpreter is this  notebook's permission boundary — the same job the `DISPATCH` dict did in notebook 1,  doing considerably more work.

In [ ]:
TASK = (    "The tests in test_buggy.py are failing. Read buggy.py, find the bug, fix it, "    "and run the tests until they pass. Then report in one sentence what the bug was.")agent = CodeAgent(    tools=[read_file, edit_file, run_tests],    model=model,    max_steps=8,    verbosity_level=2,)print("tools the agent has:", list(agent.tools))   # note the one you did not supply

## 5. Run it

In [ ]:
result = agent.run(TASK)print("\n=== result ===")print(result)

## 6. Read the code it wroteThis is the cell that makes the notebook worth doing. Every other framework in this repowould show you a list of JSON tool calls. Here you get the actual Python the modelcomposed, step by step.Look for things a JSON-calling agent could not have done in one step: calling two tools insequence, using the result of one as the argument to another, storing something in avariable, a loop, an `if`.

In [ ]:
from smolagents.memory import ActionStepfor step in agent.memory.steps:    if isinstance(step, ActionStep) and step.code_action:        print(f"--- step {step.step_number} " + "-" * 50)        print(step.code_action.strip())        if step.observations:            print("   observations:", str(step.observations)[:300].replace("\n", " "))        print()

## 7. Did it actually change the file?

In [ ]:
print(Path("buggy.py").read_text())print(subprocess.run(    [sys.executable, "-m", "pytest", "-q"],    capture_output=True, text=True, cwd=WORKDIR,).stdout)

## 8. Five frameworks, one loop| | By hand | Agent Framework | CrewAI | LangGraph | smolagents ||---|---|---|---|---|---|| The action is | JSON | JSON | JSON | JSON | **Python** || The loop is | yours | inside `run()` | inside `kickoff()` | a graph you draw | inside `run()` || Param descriptions | your JSON | `Annotated` | `Field` | `Args:` + flag | `Args:`, required || Turn cap | `max_turns=8` | an unread default | `max_iter`, default 25 | `recursion_limit`, in steps | `max_steps` || Seeing what happened | your `print` | middleware | `step_callback` | `stream()` | the code itself || Permission boundary | `DISPATCH` dict | `tools=[...]` | `tools=[...]` | `ToolNode(TOOLS)` | tools **+ a sandboxed interpreter** |Four of the five columns are the same idea in different packaging. The fifth changes whatan action *is* — and notice that this is the only column where the permission boundary hadto become something more than a list, because "run this Python" is a much larger requestthan "call this function with these arguments".### The trade**For:** fewer round trips, so less latency and fewer tokens; the model can expresscontrol flow instead of asking permission four times; composing tools is free.**Against:** you are executing model-written code. smolagents restricts the interpreter andblocks imports by default, which is real protection, but the honest position is that thisis a larger attack surface than JSON tool calls and it deserves a sandbox — smolagentssupports running the executor in Docker or a remote sandbox for exactly this reason. It isalso harder to audit: a malformed JSON call fails loudly, while wrong-but-valid Pythondoes something you did not intend.### Where this leaves youYou have now built the same agent five times. The loop has been yours, hidden twice,drawn as a graph, and replaced by a Python interpreter — and the bug it fixes has neverchanged.That is the argument for learning the mechanism rather than an API. Two of the frameworksthat used to be standard in this space were deprecated during the eighteen months beforethis workshop was written. The loop was not.### Try this1. Delete `run_tests` from the tools list. Fifth time. Predict the outcome before running.2. Ask it to do something requiring composition — *"count how many functions in buggy.py   have a mutable default argument"* — and compare how many steps it takes against what a   JSON-calling agent would need.3. Try `additional_authorized_imports=["os"]` and consider what you just permitted.

---# Instructor notes**Where this sits.** Last, and it works as the closing argument for the whole workshop.Everything before it varied the packaging; this varies the idea. If you only have time forthree notebooks total, the sequence 1 → 2 → 5 is stronger than 1 → 2 → 3, because thecontrast is sharper.Budget 30–40 minutes. Students write almost no code, so the time goes into reading whatthe agent wrote and discussing the trade.**What students write.** Just the `CodeAgent` construction. Deliberately trivial — thework in this notebook is reading, not typing. Cell 6 is where the session actually happens.**Cell 6 is the payoff.** Put a transcript from notebook 1 next to `step.code_action`output on screen. One is a list of JSON dictionaries; the other is a Python program themodel wrote to solve the problem. Ask which one is a more natural way to express *read thefile, find the bug, patch it, run the tests* — most students will say the code, and thenyou can ask what that costs.**The injected `final_answer` tool.** Small but genuinely useful: students supplied threetools and the agent has four. It punctures the idea that they fully control the actionspace just because they passed a list, and it is a nice miniature of the general pointthat frameworks make decisions you did not.**On security, be straight with them.** This is the notebook where "the dispatch table is apermission boundary" gets its hardest test. smolagents runs model-written code in arestricted interpreter with an import allowlist, which is meaningfully safer than `exec`,and it supports Docker and remote sandbox executors for when that is not enough. It isstill a bigger surface than JSON tool calls. Say plainly that you would not point this at aproduction filesystem without a container, and that the reason to teach it anyway is thatcode-as-action is a real research direction, not a toy.The `additional_authorized_imports=["os"]` exercise in section 8 is the best five minuteshere — ask what a model could do with `os` before anyone runs it.**Fifth spelling of parameter descriptions.** By now students should be able to predictthat each framework has its own, and to state why that is evidence for the workshop'sthesis rather than against it. smolagents refusing to build a tool without an `Args:`section is the only framework of the five that fails loudly, which is worth crediting.**Comparison table.** The five-column table in section 8 is your closing slide. Read thefirst row across: JSON, JSON, JSON, JSON, Python. Then the last row, where the boundary hasto grow because the request got bigger.**If the model struggles.** Code-writing is harder than emitting a tool call, and Nemotron3.5 Lightning is a small execution model. This is the notebook most likely to need astronger model:```pythonMODEL = "openai/gpt-4o-mini"```Expect more variance here than in the other four. That is worth naming out loud ratherthan hiding: the reason code-as-action is not yet the default is precisely that it asksmore of the model.**One environment note.** smolagents' `@tool` decorator reads the function's source with`inspect.getsource`, which works in a notebook cell but fails in a bare REPL. If a studentgets `OSError: could not get source code`, they are running it somewhere withoutretrievable source, not hitting a bug.